# Leksjon 13 - Agentminne


## Oppsett

Denne notatboken demonstrerer hvordan du bygger en reisebestillingsagent med **vedvarende minne** ved hjelp av **Microsoft Agent Framework** (MAF).

Du vil lære hvordan forskjellige typer agentminne — arbeidsminne, korttidshukommelse og langtidshukommelse — påvirker hvordan en agent beholder og bruker informasjon gjennom samtaler.

**Forutsetninger:**
- Et Azure AI Foundry-prosjekt med en distribuert chatmodell (f.eks. `gpt-4o-mini`).
- Logget inn med Azure CLI — kjør `az login` i terminalen din.
- `AZURE_AI_PROJECT_ENDPOINT` — endepunktet for Azure AI Foundry-prosjektet ditt.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — navnet på den distribuerte modellen din.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typer av agentminne

AI-agenter kan utnytte forskjellige typer minne, som hver tjener et bestemt formål:

### Arbeidsminne
Selve samtaletråden — meldingene som utveksles i en enkelt økt. Agenten kan referere tilbake til tidligere meldinger i samme tråd for å opprettholde sammenheng. I MAF opprettes dette via **`agent.create_session()`**, som returnerer en `AgentSession`.

### Korttidsminne
Informasjon som vedvarer i løpet av en oppgave eller økt, men som ikke lagres permanent. For eksempel kan agenten samle fakta under en flertur planleggingssamtale og bruke dem til å produsere en endelig reiserute.

### Langtidsminne
Preferanser og fakta som vedvarer **på tvers av økter**. En tilbakevendende bruker bør ikke måtte gjenta sine kostholdsrestriksjoner eller reisevaner. Langtidsminne støttes vanligvis av en ekstern lagring — en database, fil eller vektorindeks — og gjøres tilgjengelig for agenten via verktøy.


## Arbeidsminne med økter

Den enkleste formen for minne er samtaleøkten. Når du sender den samme økten (opprettet via `agent.create_session()`) til påfølgende `agent.run()`-anrop, ser agenten hele historikken for den samtalen og kan huske tidligere detaljer.

La oss lage en reiseagent og demonstrere arbeidsminne.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agenten husket riktig budsjettet fordi begge meldingene deler samme økt. Dette er **arbeidsminne** — det eksisterer kun i løpet av øktens levetid.

### Hva skjer med en ny tråd?

Hvis vi oppretter en **ny** økt, har agenten ingen hukommelse om den forrige samtalen:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Langtidsminnemønster

For å huske brukerpreferanser **på tvers av økter**, trenger vi et vedvarende lager som eksisterer utenfor samtaletråden. Agenten får tilgang til dette lageret gjennom **verktøy** — funksjoner den kan kalle for å lagre og hente informasjon.

Nedenfor implementerer vi et enkelt preferanselager i minnet (i produksjon vil du støtte dette med en database eller vektorindeks) og eksponerer det som verktøy agenten kan bruke.

### Arkitektur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Førstegangsbruker bestiller jubileumsreise

Sarah besøker for første gang. Agenten skal lagre hennes preferanser via verktøyene og bruke dem til å anbefale hoteller.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah kommer tilbake uker senere

Sarah starter en **helt ny tråd** (simulerer en ny økt). Arbeidsminnet er tomt, men lagringsstedet for langtidspreferanser har fortsatt informasjonen hennes. Agenten bør hente den og bruke den til å tilpasse anbefalingene.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sammendrag

I denne leksjonen lærte du om tre typer agentminne og hvordan du implementerer dem med Microsoft Agent Framework:

| Minnetype | MAF-mekanisme | Levetid |
|---|---|---|
| **Arbeidsminne** | `agent.create_session()` | Enkel samtale |
| **Korttidsminne** | Akkumulert kontekst innen en tråd | Enkelt oppdrag / økt |
| **Langtidsminne** | Ekstern lagring tilgjengelig via `@tool`-funksjoner | På tvers av økter |

### Viktige punkter
1. **`agent.create_session()`** gir arbeidsminne — agenten ser hele samtalehistorikken innen en økt.
2. **Nye økter mister kontekst** — uten langtidsminne kan ikke agenten huske tidligere samtaler.
3. **`@tool`-funksjoner** bygger bro — de lar agenten lagre og hente informasjon fra en vedvarende lagring.
4. **Personalisering blir bedre over tid** — jo flere preferanser som lagres, desto bedre blir agentens anbefalinger.

### Anvendelser i praksis
- **Kundeservice**: Husk kundehistorikk og preferanser
- **Personlige assistenter**: Oppretthold kontekst over dager eller uker
- **Helsevesen**: Følg opp pasientinformasjon og preferanser
- **E-handel**: Personlig tilpasset shopping basert på historikk

### Neste steg
- Erstatt det midlertidige ordbokslageret med en database eller vektorlagring (f.eks. Azure AI Search)
- Legg til minneutløp for tidskritisk informasjon
- Bygg multi-agent systemer med delt minne
- Utforsk Cognee-notatboken for kunnskapsgraf-basert minne


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettingstjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi tilstreber nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det originale dokumentet på det opprinnelige språket bør betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
